# IMPORTS

In [85]:
import glob
import pandas as pd
import os
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np
from sklearn import tree
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier,export_graphviz, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import neurokit2 as nk

# FILTERS

In [86]:
def butter_lowpass_filter(data, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

def butter_highpass_filter(data, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='highpass', analog=False)
    y = filtfilt(b, a, data)
    return y

def bandpass_filter(data, lowcut, highcut, fs, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data)

# Feature Extraction

In [87]:
def feature_extraction(data, state):
    df = data['filtered_EA'].dropna().values    
    return {
        'mean': [np.mean(df)],
        'std': [np.std(df)],
        'max': [np.max(df)],
        'min': [np.min(df)],
        'median': [np.median(df)],
        'label': state
    }

# EA Detection

In [ ]:
def ea_detection(csv_file_path):
    # Load data
    df = pd.read_csv(csv_file_path)
    df['time[s]'] = (df['LocalTimestamp'] - df['LocalTimestamp'].iloc[0])

    df = df.loc[(df['time[s]'] > 120) & (df['time[s]']  < (df['time[s]'].iloc[-1])-120)]

   
    ea_raw = df['EA'].astype(float)
  

    time = df['time[s]']
    sampling_rate = 15          
    
    ea_filtered = bandpass_filter(ea_raw, 0.1, 5, sampling_rate)

    df['filtered_EA'] = ea_filtered

    signals, info = nk.eda_process(ea_filtered, sampling_rate=15, method='neurokit', report=None)

    # nk.eda_plot(signals, info)

    return signals[['SCR_Onsets', 'SCR_Peaks',  'SCR_Height' , 'SCR_Amplitude', 'SCR_RiseTime', 'SCR_Recovery', 'SCR_RecoveryTime']]


# ML Model

In [ ]:
def pred_tree(frames):
    X = frames.drop('label', axis=1) #drops 'label' column
    y = frames['label']
    
    #splits into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    dt_model = DecisionTreeClassifier(criterion='entropy', max_depth=3)

    #trains model?
    dt_model.fit(X_train, y_train)

    #predicts 'y' values with test 'x' values
    y_pred = dt_model.predict(X_test)

    #checks accuracy of predicted 'y' against true 'y'
    acc = accuracy_score(y_test, y_pred)

    # confusion matrix
    dt_cm = confusion_matrix(y_test, y_pred, labels=dt_model.classes_)

    # precision, recall, f1 score
    print(classification_report(y_test, y_pred))

    print("Decision Tree Accuracy on set:", acc)

    print("--------------------------------------------")

    rf_model = RandomForestClassifier(n_estimators=10, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)

    print(classification_report(y_test, rf_pred))
    print("Random Forest Accuracy:", rf_model.score(X_test, y_test))

    # individual_tree = rf_model.estimators_[1]  # Get the first tree (you can choose any index)

    # plt.figure(figsize=(12, 8))
    # plot_tree(individual_tree, feature_names=['SCR_Onsets', 'SCR_Peaks',  'SCR_Height' , 'SCR_Amplitude', 'SCR_RiseTime', 'SCR_Recovery', 'SCR_RecoveryTime'], class_names=['engaged', 'relaxed'], filled=True)
    # plt.show()



In [90]:
# Run detection

engaged_filenames = glob.glob("engaged/*.csv")

data = pd.DataFrame()

for file in engaged_filenames:
    df = ea_detection(file)
    df['label'] =  'engaged'
    data = pd.concat([data, pd.DataFrame(df)], ignore_index = True)

relaxed_filenames = glob.glob("relaxed/*.csv")

for file in relaxed_filenames:
    df = ea_detection(file)
    df['label'] = 'relaxed'
    data = pd.concat([data, pd.DataFrame(df)], ignore_index = True)

pred_tree(data)

              precision    recall  f1-score   support

     engaged       0.52      0.99      0.68     12308
     relaxed       0.83      0.04      0.07     11814

    accuracy                           0.52     24122
   macro avg       0.67      0.52      0.38     24122
weighted avg       0.67      0.52      0.38     24122

Decision Tree Accuracy on set: 0.5248321034740071
--------------------------------------------
              precision    recall  f1-score   support

     engaged       0.52      0.99      0.68     12308
     relaxed       0.80      0.04      0.08     11814

    accuracy                           0.53     24122
   macro avg       0.66      0.52      0.38     24122
weighted avg       0.66      0.53      0.39     24122

Random Forest Accuracy: 0.5259514136472929
